# Many-to-Many Relations in Spotify Database

This notebook explores the many-to-many relationships between artists, albums, and tracks.

In [1]:
from pathlib import Path
import pandas as pd
from src.db import get_connection

DATABASE_PATH = Path.home() / "HDD/Datasets/annas_archive_spotify_2025_07_merged/spotify_merged.sqlite3"

In [2]:
conn = get_connection(DATABASE_PATH)

## Many to many relations

Multiple artists to album are in a many to many relation.

In [3]:
album = pd.read_sql(
    "SELECT * FROM albums WHERE name ='Mos Def & Talib Kweli Are Black Star'",
    conn,
)
album = album.sort_values("popularity", ascending=False)[:1]
album_rowid = album["rowid"].item()
album

,rowid,id,fetched_at,name,album_type,available_markets_rowid,external_id_upc,copyright_c,copyright_p,label,popularity,release_date,release_date_precision,total_tracks,external_id_amgid
0,2805,6GRzmk9UGL7odxprOPop1Q,1741824000000,Mos Def & Talib Kweli Are Black Star,album,267,00008811289720,© 2002 Rawkus Entertainment LLC,℗ 2002 Rawkus Entertainment LLC,Rawkus Entertainment,55,1998-09-29,day,13,None


For some reason the rows in `artist_album` appears to always be duplicated.

In [4]:
artists_album = pd.read_sql(
    f"SELECT * FROM artist_albums WHERE album_rowid = {album_rowid}",
    conn,
)
artists_album

,artist_rowid,album_rowid,is_appears_on,is_implicit_appears_on,index_in_album
0,1496150,2805,0,0,0.0
1,6986030,2805,1,0,NaN
2,1034160,2805,1,0,NaN
3,984013,2805,1,0,NaN
4,5169074,2805,1,0,NaN
5,1791326,2805,1,0,NaN
6,5028550,2805,1,0,NaN
7,5323826,2805,1,0,NaN
8,1496150,2805,0,0,0.0
9,6986030,2805,1,0,NaN


In [5]:
artists_album_string = ", ".join(artists_album["artist_rowid"].map(str))
artists = pd.read_sql(
    f"SELECT * FROM artists WHERE rowid IN ({artists_album_string})",
    conn,
)
artists

,rowid,id,fetched_at,name,followers_total,popularity
0,984013,6h1xNpii2MAThsl3TZfGHI,1743033600000,Punchline,727,23
1,1034160,6de0XwbJtLyirUZUqte7aD,1743033600000,Wordsworth,11919,30
2,1496150,67ei8ib6PLT1w3OkhIb4fB,1743033600000,Black Star,448425,51
3,1791326,5nLZPt6FYwWyE2aF4RvN40,1743033600000,Apani Emcee,122,24
4,5028550,2GHclqNVjqGuiE5mA7BEoc,1743033600000,Common,1009160,62
5,5169074,26W2arAlXmTgLeLdJ2nwSq,1743033600000,Jane Doe,1135,26
6,5323826,1vsWTWAvfdqNeFmXq72SlC,1743033600000,Vinia Mojica,2641,46
7,6986030,05BX9gTvlalkzuFVg3CqyL,1743638400000,Weldon Irvine,38131,41


The same is true for tracks.

In [6]:
tracks = pd.read_sql(
    f"SELECT * FROM tracks WHERE album_rowid = {album_rowid}",
    conn,
)
tracks

,rowid,id,fetched_at,name,preview_url,album_rowid,track_number,external_id_isrc,popularity,available_markets_rowid,disc_number,duration_ms,explicit
0,25985,6Rfc5vnh8qine5VneEpy7a,1743638400000,Intro,None,2805,1,USRW50200101,35,267,1,71613,1
1,25986,1mIpDPCehnf3EPk6J4ZCUF,1743033600000,Astronomy (8th Light),None,2805,2,USRW50200102,52,267,1,203680,1
2,25987,4C7Ss9bTPOWJMh3rarF1mN,1743033600000,Definition,None,2805,3,USRW50200103,56,267,1,206493,1
3,25988,5ED4wu4LKC0iNR28iMtAyv,1743033600000,Re:Definition,None,2805,4,USRW50200104,49,267,1,182973,1
4,25989,2gR8M4j8V1n0ScoE6pB3Wv,1743033600000,Children's Story,None,2805,5,USRW50200105,42,267,1,212600,1
5,25990,3Mz4AoWIedMfzCib1LYwMZ,1743033600000,Brown Skin Lady,None,2805,6,USRW50200106,50,267,1,346666,1
6,25991,42Ukmio4yQyjiZrqpys8XY,1743638400000,B Boys Will B Boys,None,2805,7,USRW50200107,35,267,1,156026,1
7,25992,3DC6AWwtPhlhsBOm4pFkt1,1743033600000,K.O.S. (Determination),None,2805,8,USRW50200108,43,267,1,289200,1
8,25993,0aoHxg1dcxHSgMJFE4umoQ,1742428800000,Hater Players,None,2805,9,USRW50200109,41,267,1,247800,1
9,25994,6W8xPUq3muUFRnxVcVjdIK,1743638400000,Yo Yeah,None,2805,10,USRW50200110,36,267,1,70000,1


In [7]:
track_artists = pd.read_sql(
    f"SELECT * FROM track_artists WHERE track_rowid=25986",
    conn,
)
track_artists

,track_rowid,artist_rowid
0,25986,1496150
1,25986,6986030


In [8]:
tracks_artist_string = ", ".join(track_artists["artist_rowid"].map(str))
artists = pd.read_sql(
    f"SELECT * FROM artists WHERE rowid IN ({tracks_artist_string})",
    conn,
)
artists

,rowid,id,fetched_at,name,followers_total,popularity
0,1496150,67ei8ib6PLT1w3OkhIb4fB,1743033600000,Black Star,448425,51
1,6986030,05BX9gTvlalkzuFVg3CqyL,1743638400000,Weldon Irvine,38131,41


Multiple genres may be associated to a single artist too.

In [9]:
genres = pd.read_sql(
    f"SELECT * FROM artist_genres WHERE artist_rowid IN ({tracks_artist_string})",
    conn,
)
genres

,artist_rowid,genre
0,1496150,east coast hip hop
1,1496150,hip hop
2,1496150,jazz rap
3,6986030,jazz funk
